# 004 合订本：R1 排序 → R2 融合 → R3 训练（Colab T4，一次跑通）

前置（Drive `/content/drive/MyDrive/Colab Notebooks/enveda/` 内放好 7 个文件）：
- `reference-fpnet-v32.py`、`np_forward_sample.json`、`np_queries_1184.json`、
  `train_structures.jsonl`（23MB）、`train_shard_200k_p0..p3.parquet`。
按 cell 顺序逐格运行；R3 训练格约 10 分钟（含 6 次 NP 评估），其余格共约 15 分钟。
产物全部写 `/content/004-output/`，最后统一下载回本地 `artifacts/004-research/`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/Colab Notebooks/enveda')
OUT = Path('/content/004-output'); OUT.mkdir(parents=True, exist_ok=True)
need = ['reference-fpnet-v32.py','np_forward_sample.json','np_queries_1184.json',
        'train_structures.jsonl'] + ['train_shard_200k_p%d.parquet'%i for i in range(4)]
missing = [f for f in need if not (DATA/f).exists()]
print('missing:', missing)
assert not missing, missing
print('all inputs present')


In [ ]:
import importlib.util, subprocess, sys
missing = [n for n in ('torch','numpy','pyarrow','kagglehub','rdkit')
           if importlib.util.find_spec(n) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing], check=True)
print({'installed_missing': missing})
import platform, torch
print('python =', sys.version.split()[0])
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,3), 'GiB')
import pyarrow as pa
print('pyarrow =', pa.__version__)
import rdkit
print('rdkit =', rdkit.__version__)


In [ ]:
import kagglehub, hashlib, time
from pathlib import Path
for h in ['prvsiyan/casmi26-fp-models-v2/versions/1',
          'prvsiyan/coconut-casmi26-candidates/versions/2']:
    t0=time.perf_counter(); p=kagglehub.dataset_download(h)
    print(h, '->', p, round(time.perf_counter()-t0,1))
W = Path('/root/.cache/kagglehub/datasets/prvsiyan/casmi26-fp-models-v2/versions/1/fp_single_s2.pt')
print('single bytes:', W.stat().st_size)
assert W.stat().st_size == 144110255
B = Path('/root/.cache/kagglehub/datasets/prvsiyan/coconut-casmi26-candidates/versions/2/fp_bits.npy')
print('fp_bits bytes:', B.stat().st_size)
assert B.stat().st_size == 55568


In [ ]:
import numpy as np
from rdkit import Chem, RDLogger
import rdkit
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Descriptors import ExactMolWt
RDLogger.DisableLog('rdApp.*')
BITS = np.load('/root/.cache/kagglehub/datasets/prvsiyan/coconut-casmi26-candidates/versions/2/fp_bits.npy')
print('BITS:', BITS.shape, BITS.dtype, int(BITS.min()), int(BITS.max()))
assert BITS.shape==(6930,) and int(BITS.min())>=0 and int(BITS.max())<10407
m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
m3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
rk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)
def fp_and_mass(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    fp = np.concatenate([m2.GetFingerprintAsNumPy(m).astype(np.uint8),
                         m3.GetFingerprintAsNumPy(m).astype(np.uint8),
                         rk.GetFingerprintAsNumPy(m).astype(np.uint8),
                         np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[BITS]
    return fp, float(ExactMolWt(m))
for smi in ['CCO','CC(=O)O','Oc1ccccc1','CN1C=NC2=C1C(=O)N(C)C(=O)N2C','OCC(O)CO']:
    r = fp_and_mass(smi)
    print(smi, '->', None if r is None else (r[0].shape, int(r[0].sum()), round(r[1],4)))


In [ ]:
import importlib.util, json, torch
spec = importlib.util.spec_from_file_location('refv32', (DATA/'reference-fpnet-v32.py'))
ref = importlib.util.module_from_spec(spec); spec.loader.exec_module(ref)
ck = torch.load('/root/.cache/kagglehub/datasets/prvsiyan/casmi26-fp-models-v2/versions/1/fp_single_s2.pt', map_location='cpu', weights_only=True)
net = ref.FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().eval()
net.load_state_dict(ck['model'], strict=True)
print('model ok:', ck['nbits'], ck['d'], ck['layers'], '| params:', sum(p.numel() for p in net.parameters()))
samp = json.load(open((DATA/'np_forward_sample.json')))['spectra'][:4]
print('sanity spectra:', len(samp))


## 训练结构池（275,810 全量，约 4 分钟）：真值必在池内，R1/R3 共用

In [ ]:
import json
import numpy as np
from multiprocessing import Pool as MPool
recs = [json.loads(l) for l in open((DATA/'train_structures.jsonl'))]
print('structures:', len(recs))
assert len(recs) == 275810
def _init():
    global _m2,_m3,_rk
    _m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _m3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _rk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)
def one(r):
    import numpy as _np
    from rdkit import Chem
    m = Chem.MolFromSmiles(r['smiles'])
    if m is None: return None
    try:
        fp = _np.concatenate([_m2.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _m3.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _rk.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _np.array(MACCSkeys.GenMACCSKeys(m), dtype=_np.uint8)])[BITS].astype(_np.float32)
        return (r['inchikey14'], fp, float(ExactMolWt(m)))
    except Exception:
        return None
Fp, Mm, Kk, fail = [], [], [], 0
with MPool(4, initializer=_init) as p:
    for i in range(0, len(recs), 25000):
        out = p.map(one, recs[i:i+25000], chunksize=500)
        Fp.append(np.packbits(np.array([x[1] if x else np.zeros(6930,np.uint8) for x in out], dtype=np.uint8), axis=1))
        Mm.append(np.array([x[2] if x else np.nan for x in out], dtype=np.float64))
        Kk += [x[0] if x else '' for x in out]
        fail += sum(1 for x in out if x is None)
        print('pool done %d/%d fail=%d' % (min(i+25000,len(recs)), len(recs), fail))
pool_fp = np.vstack(Fp); pool_mass = np.concatenate(Mm); pool_keys = np.array(Kk)
np.save('/content/pool_fp.npy', pool_fp); np.save('/content/pool_mass.npy', pool_mass); np.save('/content/pool_keys.npy', pool_keys)
print(pool_fp.shape, pool_mass.shape, pool_keys.shape, 'fail=', fail)
key_to_idx = {k:i for i,k in enumerate(pool_keys) if k}
print('indexed:', len(key_to_idx))


## R1：NP1184 全量（mass 窗 + FPNet，打分约 10 秒）

In [ ]:
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
    S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
    Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {'[M+H]+':(1,1,PROTON),'[M+NH4]+':(1,1,NH4-E),"[M+Na]+":(1,1,MASS['Na']-E),
 '[M+K]+':(1,1,MASS['K']-E),'[M-H2O+H]+':(1,1,PROTON-H2O),'[M-2H2O+H]+':(1,1,PROTON-2*H2O),
 '[M+2H]2+':(1,2,2*PROTON),'[M]+':(1,1,-E),'[M-H2O]+':(1,1,-E-H2O),
 '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 '[M-H]-':(1,1,-PROTON),'[M-H2O-H]-':(1,1,-PROTON-H2O),'[M+CH2O2-H]-':(1,1,FORMATE-PROTON),
 '[M+C2H4O2-H]-':(1,1,ACETATE-PROTON),"[M+Cl]-":(1,1,MASS['Cl']+E),'[M]-':(1,1,E),
 '[M-2H]-':(1,2,-2*PROTON),'[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON),
 '[2M+H]+':(2,1,PROTON),'[2M+Na]+':(2,1,MASS['Na']-E),'[2M+NH4]+':(2,1,NH4-E),
 '[2M+K]+':(2,1,MASS['K']-E),'[2M-H]-':(2,1,-PROTON),'[2M+CH2O2-H]-':(2,1,FORMATE-PROTON),
 '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON),'[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
 '[3M+H]+':(3,1,PROTON),'[3M-H]-':(3,1,-PROTON)}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m]*z - d)/n
    return out
Q = json.load(open((DATA/'np_queries_1184.json')))
print('Q:', len(Q))
@torch.no_grad()
def single_z(row):
    import numpy as _np
    a,b = ref.prep_peaks(_np.asarray(row['ms2_mzs'],float), _np.asarray(row['ms2_normalized_intensities'],float), float(row['precursor_mz']))
    N = len(a)
    mz=_np.zeros((1,N),_np.float32); it=_np.zeros((1,N),_np.float32); pad=_np.zeros((1,N),bool)
    mz[0,:N]=a; it[0,:N]=b
    ce = row.get('collision_energy_ev')
    ce = float(_np.mean(_np.atleast_1d(ce))) if ce else 25.0
    T = lambda x: torch.as_tensor(x, device='cuda')
    args=(T(mz),T(it),T(pad),T(_np.array([float(row['precursor_mz'])],_np.float32)),
          T(_np.array([ref.ADDUCT_IX.get(row['adduct'],ref.ADDUCT_IX['<unk>'])])),
          T(_np.array([ref.instr_family(row['instrument_type'])])),
          T(_np.array([ce],_np.float32)),
          T(_np.array([1.0 if row['ionization_mode']=='positive' else -1.0],_np.float32)))
    return net(*args).float().cpu().numpy()[0]
def window(qm, ppm=20.0):
    return np.where(np.abs(pool_mass - qm) <= qm*ppm*1e-6)[0]
def rank_in(z, w, tikey):
    F = np.unpackbits(pool_fp[w], axis=1)[:, :6930].astype(np.float32)
    order = np.argsort(-(F @ z))
    ti = key_to_idx.get(tikey)
    if ti is None: return None
    pos = np.where(w[order] == ti)[0]
    return int(pos[0])+1 if len(pos) else None
res = []
import time as _t; _t0 = _t.perf_counter()
for i,row in enumerate(Q):
    qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
    w = window(qm)
    r = rank_in(single_z(row), w, row['inchikey14']) if len(w) else None
    res.append({'id':i,'key':row['inchikey14'],'win':int(len(w)),'rank':r})
    if (i+1)%200==0: print(i+1, round(_t.perf_counter()-_t0,1))
json.dump(res, open('/content/004-output/r1_np1184.json','w'))
rk = [r['rank'] for r in res if r['rank']]
print('n=%d ranked=%d mrr=%.4f top1=%.4f' % (len(res), len(rk), sum(1/x for x in rk)/len(res), sum(1 for x in rk if x==1)/len(res)))


## R2：分子级融合（single / 全均值 / 同 adduct / 轮转，约 1 分钟）

In [ ]:
import json
import numpy as np
from collections import defaultdict
res1 = json.load(open('/content/004-output/r1_np1184.json'))
G = defaultdict(list)
for i,row in enumerate(Q):
    G[row['inchikey14']].append((i,row))
print('molecules:', len(G))
def rrfuse(lists, k=25):
    seen,out = set(),[]
    for r in range(max(len(l) for l in lists)):
        for l in lists:
            if r < len(l) and l[r] not in seen:
                seen.add(l[r]); out.append(l[r])
            if len(out) >= k: return out
    return out
def full_top(z, w):
    F = np.unpackbits(pool_fp[w], axis=1)[:, :6930].astype(np.float32)
    return list(w[np.argsort(-(F @ z))][:25])
out = []
for tikey,members in G.items():
    members_sorted = sorted(members, key=lambda t: len(t[1]['ms2_mzs']), reverse=True)
    zs,tops,qms = [],[],[]
    for _,row in members:
        z = single_z(row)
        zs.append(z)
        qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
        qms.append(qm)
        tops.append(full_top(z, window(qm)))
    W = np.array(sorted(set().union(*[set(window(q)) for q in qms])))
    F = np.unpackbits(pool_fp[W], axis=1)[:, :6930].astype(np.float32)
    ti = key_to_idx.get(tikey)
    def rank_of(z):
        order = np.argsort(-(F @ z))
        pos = np.where(W[order]==ti)[0]
        return int(pos[0])+1 if len(pos) else None
    by_ad = defaultdict(list)
    for (_,row),z in zip(members,zs):
        by_ad[row['adduct']].append(z)
    fused = rrfuse(tops)
    rr = (fused.index(ti)+1) if ti in fused else None
    srow = members_sorted[0][1]
    qm_s = neutral_mass(np.array([srow['precursor_mz']]), [srow['adduct']])[0]
    out.append({'key':tikey,'n_spectra':len(members),
        'single_rank': rank_in(single_z(srow), window(qm_s), tikey),
        'mean_all_rank': rank_of(np.mean(zs,axis=0)),
        'mean_ad_rank': rank_of(np.mean([np.mean(v,axis=0) for v in by_ad.values()],axis=0)),
        'rrfuse_rank': rr})
json.dump(out, open('/content/004-output/r2_np250.json','w'))
for name in ['single_rank','mean_all_rank','mean_ad_rank','rrfuse_rank']:
    rr = [1/r[name] for r in out if r[name]]
    print(name, 'mrr=%.4f top1=%.4f ranked=%d/250' % (sum(rr)/250, sum(1 for r in out if r[name]==1)/250, len(rr)))


## R3：5000 步 head-only 训练 + 每 1000 步 NP1184 评估（约 10 分钟）
训练集 20 万分片（NP 整组已排除），验证 NP1184 同 R1 口径。权重落盘。

In [ ]:
import json, time, random
import numpy as np, torch
import pyarrow.parquet as pq
import pyarrow as pa
torch.manual_seed(7); random.seed(7); np.random.seed(7)
files = [str(DATA/'train_shard_200k_p%d.parquet')%i for i in range(4)]
t = pa.concat_tables([pq.read_table(f) for f in files]).to_pydict()
N = len(t['inchikey14'])
print('train rows:', N)
smi_of = {}
for k,s in zip(t['inchikey14'], t['normalized_smiles']):
    if k not in smi_of: smi_of[k] = s
print('distinct:', len(smi_of))
from multiprocessing import Pool as MPool
def _init():
    global _m2,_m3,_rk
    _m2 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _m3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _rk = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)
def one(kv):
    import numpy as _np
    from rdkit import Chem
    k,s = kv
    m = Chem.MolFromSmiles(s)
    if m is None: return (k,None)
    try:
        fp = _np.concatenate([_m2.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _m3.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _rk.GetFingerprintAsNumPy(m).astype(_np.uint8),
                              _np.array(MACCSkeys.GenMACCSKeys(m), dtype=_np.uint8)])[BITS].astype(_np.float32)
        return (k,fp)
    except Exception:
        return (k,None)
with MPool(4, initializer=_init) as p:
    items = list(smi_of.items())
    res_t = []
    for i in range(0, len(items), 20000):
        res_t += p.map(one, items[i:i+20000], chunksize=500)
        print('tgt %d/%d' % (min(i+20000,len(items)), len(items)))
tgt = {k:v for k,v in res_t if v is not None}
print('targets:', len(tgt))
for n,p_ in net.named_parameters():
    p_.requires_grad = n.startswith('head')
net.train()
opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')
bce = torch.nn.BCEWithLogitsLoss()
def ce_of(v):
    try: return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
    except Exception: return 25.0
def make_batch(bs=32):
    idx = random.sample(range(N), bs)
    P,Tg = [],[]
    for i in idx:
        a,b = ref.prep_peaks(np.asarray(t['ms2_mzs'][i],float), np.asarray(t['ms2_normalized_intensities'][i],float), float(t['precursor_mz'][i]))
        if not len(a) or t['inchikey14'][i] not in tgt: continue
        P.append((a,b,i)); Tg.append(tgt[t['inchikey14'][i]])
    B = len(P); Np = max(len(a) for a,_,_ in P)
    mz=np.zeros((B,Np),np.float32); it=np.zeros((B,Np),np.float32); pad=np.ones((B,Np),bool)
    for j,(a,b,_) in enumerate(P):
        mz[j,:len(a)]=a; it[j,:len(b)]=b; pad[j,:len(a)]=False
    T = lambda x: torch.as_tensor(x, device='cuda')
    ii=[i for _,_,i in P]
    args=(T(mz),T(it),T(pad),T(np.array([float(t['precursor_mz'][i]) for i in ii],np.float32)),
          T(np.array([ref.ADDUCT_IX.get(t['adduct'][i],ref.ADDUCT_IX['<unk>']) for i in ii])),
          T(np.array([ref.instr_family(t['instrument_type'][i]) for i in ii])),
          T(np.array([ce_of(t['collision_energy_ev'][i]) for i in ii],np.float32)),
          T(np.array([1.0 if t['ionization_mode'][i]=='positive' else -1.0 for i in ii],np.float32)))
    return args, T(np.stack(Tg))
def train_step(bs=32):
    args,y = make_batch(bs)
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', dtype=torch.float16):
        loss = bce(net(*args), y)
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()
    return float(loss.detach())
@torch.no_grad()
def eval_np():
    net.eval()
    rr = []
    for row in Q:
        qm = neutral_mass(np.array([row['precursor_mz']]), [row['adduct']])[0]
        w = window(qm)
        r = rank_in(single_z(row), w, row['inchikey14']) if len(w) else None
        rr.append(r)
    net.train()
    got = [x for x in rr if x]
    return sum(1/x for x in got)/len(rr), sum(1 for x in got if x==1)/len(rr)
log = {'seed':7,'mode':'head_only','bs':32,'lr':3e-4,'steps':5000,'eval_every':1000,'history':[]}
m,v = eval_np()
log['history'].append({'step':0,'mrr':m,'top1':v})
print('step 0 mrr=%.4f top1=%.4f' % (m,v))
t0=time.perf_counter()
for s in range(1,5001):
    loss = train_step(32)
    if s%1000==0:
        m,v = eval_np()
        log['history'].append({'step':s,'loss':loss,'mrr':m,'top1':v,'elapsed':round(time.perf_counter()-t0,1)})
        print('step %d loss=%.4f mrr=%.4f top1=%.4f elapsed=%.0fs' % (s,loss,m,v,time.perf_counter()-t0))
torch.save({'head':{n:p.cpu() for n,p in net.named_parameters() if p.requires_grad},'config':log}, '/content/004-output/head_5000.pt')
json.dump(log, open('/content/004-output/r3_log.json','w'), indent=1)
print('saved head_5000.pt + r3_log.json')


## 收尾：下载 `/content/004-output/` 全部（`r1_np1184.json`、`r2_np250.json`、`head_5000.pt`、`r3_log.json`）回本地 `artifacts/004-research/`，把 `r3_log.json` 的 history 贴回 review。